In [1]:
# CELL 1 — Imports and project paths
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import warnings
import importlib.util

import numpy as np
import pandas as pd
import sklearn
import joblib
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)
RANDOM_STATE = 42

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents]
     if (p / "data" / "processed").is_dir()
     and (p / "notebooks").is_dir()
     and (p / "reports").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate ml-workstream-2. Expected data/processed, notebooks and reports. "
        f"Current working directory: {cwd}"
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
for folder in [CONFIG_DIR, MODELS_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

FEATURE_TABLE_PATH = PROCESSED_DIR / "participant_phase_features.csv"
MANIFEST_PATH = CONFIG_DIR / "baseline_model_feature_manifest.json"
if not FEATURE_TABLE_PATH.is_file():
    raise FileNotFoundError(f"Missing processed feature table: {FEATURE_TABLE_PATH}. Run Notebook 03.")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing feature manifest: {MANIFEST_PATH}. Run Notebook 04 completely.")

MODEL_PATH = MODELS_DIR / "eda_stress_random_forest.joblib"
METADATA_PATH = MODELS_DIR / "eda_stress_random_forest.metadata.json"

print("Project root :", PROJECT_ROOT)
print("Feature table:", FEATURE_TABLE_PATH)
print("Model output :", MODEL_PATH)
print("Python       :", platform.python_version())
print("scikit-learn :", sklearn.__version__)
print("joblib       :", joblib.__version__)

Project root : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2
Feature table: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\data\processed\participant_phase_features.csv
Model output : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\models\eda_stress_random_forest.joblib
Python       : 3.11.18
scikit-learn : 1.9.1
joblib       : 1.6.0


In [2]:
# CELL 2 — Load the exact EDA feature set used by the baseline experiment
features = pd.read_csv(FEATURE_TABLE_PATH)
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

required = {"participant_id", "phase", "target"}
missing = required - set(features.columns)
if missing:
    raise KeyError(f"Feature table is missing required columns: {sorted(missing)}")

# Use the exact feature list from Notebook 04 rather than silently creating a new one.
manifest_key = "EDA — all available rows"
if manifest_key not in manifest:
    possible = [k for k in manifest if k.lower().startswith("eda") and "all available" in k.lower()]
    if len(possible) != 1:
        raise KeyError(
            f"Could not uniquely find '{manifest_key}' in feature manifest. "
            f"Available experiments: {list(manifest)}"
        )
    manifest_key = possible[0]

feature_columns = manifest[manifest_key]["features"]
if not feature_columns:
    raise ValueError("The EDA feature manifest contains no model features.")
missing_features = [c for c in feature_columns if c not in features.columns]
if missing_features:
    raise KeyError(f"Manifest features missing from feature table: {missing_features}")

features["participant_id"] = features["participant_id"].astype("string").str.strip()
features["phase"] = features["phase"].astype("string").str.strip().str.lower()
phase_to_target = {"relax": 0, "stress": 1}
expected_target = features["phase"].map(phase_to_target)
if expected_target.isna().any():
    raise ValueError("Unexpected/missing phase labels in processed feature table.")
features["target"] = expected_target.astype(int)

for col in feature_columns:
    features[col] = pd.to_numeric(features[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

# Retain observations with at least one observed EDA input, matching Notebook 04 cohort logic.
cohort = features.loc[features[feature_columns].notna().any(axis=1)].copy().reset_index(drop=True)
if cohort.empty or cohort["target"].nunique() != 2:
    raise ValueError("Eligible EDA cohort is empty or does not contain both RELAX and STRESS classes.")
if cohort["participant_id"].isna().any() or (cohort["participant_id"] == "").any():
    raise ValueError("Missing participant IDs found; refusing to package an untraceable cohort.")

X = cohort[feature_columns].copy()
y = cohort["target"].astype(int)

print(f"Selected experiment: {manifest_key}")
print(f"Eligible observations: {len(cohort)}")
print(f"Participants: {cohort['participant_id'].nunique()}")
print("Class counts:")
display(cohort["phase"].value_counts().rename_axis("phase").to_frame("observations"))
print("Model features:", feature_columns)
print("Missing values per model feature:")
display(X.isna().sum().rename("missing_count").to_frame())

Selected experiment: EDA — all available rows
Eligible observations: 124
Participants: 64
Class counts:


,observations
phase,
relax,64
stress,60


Model features: ['eda_scl_mean_us', 'eda_scl_median_us', 'eda_scl_std_us', 'eda_scl_mad_us', 'eda_scl_iqr_us', 'eda_scl_min_us', 'eda_scl_max_us', 'eda_scl_slope_us_per_sec']
Missing values per model feature:


,missing_count
eda_scl_mean_us,0
eda_scl_median_us,0
eda_scl_std_us,0
eda_scl_mad_us,0
eda_scl_iqr_us,0
eda_scl_min_us,0
eda_scl_max_us,0
eda_scl_slope_us_per_sec,0


In [3]:
# CELL 3 — Refit the already-selected baseline configuration on all eligible EDA rows
# This is a full-cohort refit for a demo/integration artifact, NOT a new independent evaluation.
model_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=150,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=1,
        random_state=RANDOM_STATE,
    )),
])

print("Fitting EDA Random Forest on the complete eligible cohort...", flush=True)
model_pipeline.fit(X, y)
print("Fit complete.")

classes = model_pipeline.named_steps["model"].classes_.tolist()
if classes != [0, 1]:
    raise RuntimeError(f"Expected class order [0, 1], got {classes}")

joblib.dump(model_pipeline, MODEL_PATH, compress=3)
print("Saved model:", MODEL_PATH)
print("Artifact size:", f"{MODEL_PATH.stat().st_size:,} bytes")

metadata = {
    "model_id": "eda_stress_random_forest_v1",
    "model_type": "scikit-learn Pipeline(SimpleImputer, RandomForestClassifier)",
    "status": "experimental_research_demo",
    "purpose": "Classify controlled QoL_Stress RELAX/STRESS experimental phase from EDA summary features.",
    "target_mapping": {"0": "RELAX", "1": "STRESS"},
    "positive_class": "STRESS",
    "feature_columns": list(feature_columns),
    "preprocessing": {"imputer": "median", "fit_scope": "complete eligible cohort for this packaged artifact"},
    "estimator_parameters": {
        "n_estimators": 150,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "class_weight": "balanced_subsample",
        "random_state": RANDOM_STATE,
    },
    "training_data": {
        "dataset": "QoL_Stress v1.0.0",
        "feature_table": "data/processed/participant_phase_features.csv",
        "observations": int(len(cohort)),
        "participants": int(cohort["participant_id"].nunique()),
        "class_counts": {"RELAX": int((y == 0).sum()), "STRESS": int((y == 1).sum())},
    },
    "reference_loso_result": {
        "balanced_accuracy": 0.693,
        "macro_f1": 0.693,
        "source": "Notebook 04 baseline report; participant-independent LOSO out-of-fold estimate",
        "note": "Reference evaluation result, not a score measured on the full-cohort refit artifact.",
    },
    "limitations": [
        "Predicts controlled experimental phase labels, not chronic stress or a mental-health condition.",
        "Not clinically validated and not validated on astronauts or spaceflight data.",
        "Random Forest probabilities are not calibrated confidence estimates.",
        "The packaged model is fitted on all available study observations; do not report its training-set performance as test performance.",
    ],
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "joblib_version": joblib.__version__,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved metadata:", METADATA_PATH)

Fitting EDA Random Forest on the complete eligible cohort...
Fit complete.
Saved model: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\models\eda_stress_random_forest.joblib
Artifact size: 173,651 bytes
Saved metadata: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\models\eda_stress_random_forest.metadata.json


In [4]:
# CELL 4 — Generate a small application-facing inference module
# This creates project/src/eda_stress_model.py for use by the backend or desktop app.
SRC_DIR = PROJECT_ROOT / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "__init__.py").write_text("\"\"\"ML inference modules for the astronaut health demo.\"\"\"\n", encoding="utf-8")
MODULE_PATH = SRC_DIR / "eda_stress_model.py"
module_source = '"""Experimental EDA phase classifier for the Astronaut Health Monitoring demo.\n\nPredicts the controlled QoL_Stress RELAX/STRESS phase from EDA summary features.\nThis is not a medical diagnostic or astronaut-health model.\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Mapping\nimport json\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\n_PROJECT_ROOT = Path(__file__).resolve().parents[1]\n_MODEL_PATH = _PROJECT_ROOT / "models" / "eda_stress_random_forest.joblib"\n_METADATA_PATH = _PROJECT_ROOT / "models" / "eda_stress_random_forest.metadata.json"\n\n\ndef _load_bundle():\n    if not _MODEL_PATH.is_file() or not _METADATA_PATH.is_file():\n        raise FileNotFoundError(\n            "EDA model files are missing. Run notebooks/06_package_eda_model_for_product.ipynb first."\n        )\n    # Only load locally generated, trusted joblib artifacts. Never load untrusted pickle files.\n    model = joblib.load(_MODEL_PATH)\n    metadata = json.loads(_METADATA_PATH.read_text(encoding="utf-8"))\n    return model, metadata\n\n\ndef predict_eda_phase(features: Mapping[str, Any]) -> dict[str, Any]:\n    """Predict RELAX/STRESS from one EDA feature mapping.\n\n    Missing individual values are handled by the saved median imputer. If every\n    expected feature is missing/non-numeric/non-finite, no prediction is produced.\n    """\n    if not isinstance(features, Mapping):\n        raise TypeError("features must be a mapping of feature name to numeric value")\n\n    model, metadata = _load_bundle()\n    feature_names = metadata["feature_columns"]\n    row = {}\n    finite_count = 0\n    for name in feature_names:\n        value = features.get(name)\n        try:\n            number = float(value)\n            if not np.isfinite(number):\n                number = np.nan\n        except (TypeError, ValueError):\n            number = np.nan\n        if np.isfinite(number):\n            finite_count += 1\n        row[name] = number\n\n    if finite_count == 0:\n        return {\n            "status": "insufficient_data",\n            "model_id": metadata["model_id"],\n            "phase_prediction": None,\n            "stress_probability_uncalibrated": None,\n            "available_feature_count": 0,\n            "required_feature_count": len(feature_names),\n            "message": "No usable EDA model features were supplied; no prediction was produced.",\n        }\n\n    input_frame = pd.DataFrame([row], columns=feature_names)\n    predicted_class = int(model.predict(input_frame)[0])\n    probabilities = model.predict_proba(input_frame)[0]\n    classes = list(model.named_steps["model"].classes_)\n    stress_probability = float(probabilities[classes.index(1)])\n\n    return {\n        "status": "ok",\n        "model_id": metadata["model_id"],\n        "phase_prediction": "STRESS" if predicted_class == 1 else "RELAX",\n        "stress_probability_uncalibrated": stress_probability,\n        "available_feature_count": finite_count,\n        "required_feature_count": len(feature_names),\n        "message": "Experimental phase-classifier output only; not a diagnosis or medical risk score.",\n    }\n'
MODULE_PATH.write_text(module_source, encoding="utf-8")
print("Created inference module:", MODULE_PATH)
print("Import path from project root: from src.eda_stress_model import predict_eda_phase")

Created inference module: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\src\eda_stress_model.py
Import path from project root: from src.eda_stress_model import predict_eda_phase


In [5]:
# CELL 5 — Smoke-test the saved artifact and application inference function
spec = importlib.util.spec_from_file_location("eda_stress_model", MODULE_PATH)
eda_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eda_module)

# Use one real processed feature row only to verify the local inference path.
sample = X.iloc[0].to_dict()
smoke_result = eda_module.predict_eda_phase(sample)
print("Inference smoke test:")
print(json.dumps(smoke_result, indent=2))
if smoke_result["status"] != "ok" or smoke_result["phase_prediction"] not in {"RELAX", "STRESS"}:
    raise RuntimeError("Inference smoke test failed.")

# Verify all-missing input safely returns no prediction.
empty_result = eda_module.predict_eda_phase({})
print("\nInsufficient-data test:")
print(json.dumps(empty_result, indent=2))
if empty_result["status"] != "insufficient_data" or empty_result["phase_prediction"] is not None:
    raise RuntimeError("Insufficient-data handling test failed.")

print("\nBoth inference smoke tests passed.")

Inference smoke test:
{
  "status": "ok",
  "model_id": "eda_stress_random_forest_v1",
  "phase_prediction": "RELAX",
  "stress_probability_uncalibrated": 0.06851866529536069,
  "available_feature_count": 8,
  "required_feature_count": 8,
  "message": "Experimental phase-classifier output only; not a diagnosis or medical risk score."
}

Insufficient-data test:
{
  "status": "insufficient_data",
  "model_id": "eda_stress_random_forest_v1",
  "phase_prediction": null,
  "stress_probability_uncalibrated": null,
  "available_feature_count": 0,
  "required_feature_count": 8,
  "message": "No usable EDA model features were supplied; no prediction was produced."
}

Both inference smoke tests passed.


In [6]:
# CELL 6 — Write model card and application integration contract
MODEL_CARD_PATH = REPORTS_DIR / "eda_stress_model_card.md"
INTEGRATION_PATH = REPORTS_DIR / "eda_stress_model_integration_contract.md"

feature_list_md = "\n".join(f"- `{c}`" for c in feature_columns)
model_card = f"""# EDA Stress-Phase Classifier — Model Card

## Identity
- Model ID: `{metadata['model_id']}`
- Status: Experimental research/demo component
- Model: Median imputation + Random Forest classifier
- Dataset: QoL_Stress v1.0.0 (PhysioNet)
- Created (UTC): {metadata['created_at_utc']}

## Intended use
Classify the controlled experimental `RELAX` versus `STRESS` phase represented in the QoL_Stress dataset, using the listed EDA summary features. This artifact is intended for software integration and demonstration of an ML inference workflow.

## Training data
- Observations used for full-cohort refit: {len(cohort)}
- Participants represented: {cohort['participant_id'].nunique()}
- RELAX observations: {int((y == 0).sum())}
- STRESS observations: {int((y == 1).sum())}

## Reference evaluation
The preceding participant-independent LOSO baseline reported balanced accuracy 0.693 and macro-F1 0.693 for EDA + Random Forest. These are out-of-fold cross-validation results from Notebook 04, not performance measured on this full-cohort refit. They are not evidence of clinical or spaceflight validity.

## Inputs
{feature_list_md}

## Outputs
- `phase_prediction`: `RELAX` or `STRESS`
- `stress_probability_uncalibrated`: raw Random Forest class probability; not calibrated confidence and not a medical risk score
- `status`: `ok` or `insufficient_data`

## Important limitations
- Not a chronic-stress detector, psychiatric assessment, medical diagnostic, or treatment recommendation.
- Not trained or validated on astronauts or spaceflight data.
- Small controlled-study cohort; external generalization is unknown.
- Random Forest probabilities are uncalibrated.
- Inputs must be generated with the same feature definitions, units, and preprocessing semantics as the source feature table.
- Do not infer EDA features from unrelated measurements or fill missing sensor data with invented values.

## Dataset citation
Calvo, A., Cejudo, A., & Martín Andonegui, C. (2026). *QoL_Stress: A Multimodal Dataset of Physiological and Self-Reported Stress Responses* (version 1.0.0). PhysioNet. https://doi.org/10.13026/vqtz-pm70
"""
MODEL_CARD_PATH.write_text(model_card, encoding="utf-8")

integration_contract = f"""# EDA Model Integration Contract

## Import
From the project root:

```python
from src.eda_stress_model import predict_eda_phase
```

## Function
`predict_eda_phase(features: Mapping[str, number | None]) -> dict`

## Required feature keys
{feature_list_md}

Pass only features computed from actual EDA observations using the source feature definitions and compatible units. Individual features may be omitted or set to `None`; the saved median imputer handles missing individual inputs. If no expected feature has a usable finite numeric value, the function returns `status='insufficient_data'` and no phase prediction.

## Response example

```json
{{
  "status": "ok",
  "model_id": "eda_stress_random_forest_v1",
  "phase_prediction": "STRESS",
  "stress_probability_uncalibrated": 0.61,
  "available_feature_count": 6,
  "required_feature_count": 8,
  "message": "Experimental phase-classifier output only; not a diagnosis or medical risk score."
}}
```

The values above are illustrative response structure, not a model result.

## UI/API handling rules
- Label this component as an experimental EDA phase classifier.
- Display the prediction as a dataset-phase classification, not as an astronaut stress diagnosis.
- Never label `stress_probability_uncalibrated` as confidence, risk, or clinical probability.
- Preserve `insufficient_data`; do not substitute a default prediction.
- Store feature provenance, sensor timestamp, device/source identifier, model ID, and inference timestamp with any application record.
- The app must continue functioning when the model is unavailable; treat inference as optional decision-support content.
"""
INTEGRATION_PATH.write_text(integration_contract, encoding="utf-8")

print("Saved:")
for path in [MODEL_PATH, METADATA_PATH, MODULE_PATH, MODEL_CARD_PATH, INTEGRATION_PATH]:
    print(" -", path)
print("\nPackaging complete. No new evaluation, tuning, or clinical validation was performed.")

Saved:
 - D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\models\eda_stress_random_forest.joblib
 - D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\models\eda_stress_random_forest.metadata.json
 - D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\src\eda_stress_model.py
 - D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\reports\eda_stress_model_card.md
 - D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\reports\eda_stress_model_integration_contract.md

Packaging complete. No new evaluation, tuning, or clinical validation was performed.
